Import Required Libraries

In this step, we import all the libraries required for data processing, text preprocessing, tokenization, sequence padding, model creation, and visualization.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

| Library          | Purpose                          |
| ---------------- | -------------------------------- |
| NumPy            | Numerical computations           |
| Pandas           | Dataset loading and manipulation |
| Matplotlib       | Data visualization               |
| TensorFlow       | Deep Learning framework          |
| Keras            | Build Seq2Seq model              |
| Embedding        | Convert words into dense vectors |
| LSTM             | Learn sentence sequences         |
| Dense            | Predict next word                |
| Tokenizer        | Convert text into integers       |
| pad_sequences    | Make all sentence lengths equal  |
| train_test_split | Split training and testing data  |


Load Dataset

In [4]:
import pandas as pd

dataset = pd.read_csv("Dataset_English_Hindi.csv", encoding='latin1')

Display Dataset Shape

In [5]:
dataset.shape

(130476, 2)

130476 sentence pairs
2 columns

Display Dataset Columns

In [6]:
dataset.columns

Index(['English', 'Hindi'], dtype='object')

Display Dataset Information

In [7]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130164 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


Data Cleaning

In [8]:
# Check missing values
dataset.isnull().sum()
# Remove missing values
dataset = dataset.dropna()
# Remove duplicate rows
dataset = dataset.drop_duplicates()
# Reset index
dataset = dataset.reset_index(drop=True)
# Check dataset shape
dataset.shape
# Display first 5 rows
dataset.head()

,English,Hindi
0,Help!,à¤¬à¤à¤¾à¤!
1,Jump.,à¤à¤à¤²à¥.
2,Jump.,à¤à¥à¤¦à¥.
3,Jump.,à¤à¤²à¤¾à¤à¤.
4,Hello!,à¤¨à¤®à¤¸à¥à¤¤à¥à¥¤


Add Start and End Tokens

In [9]:
dataset["Hindi"] = dataset["Hindi"].apply(
    lambda x: "<start> " + x + " <end>"
)
dataset.head()

,English,Hindi
0,Help!,<start> à¤¬à¤à¤¾à¤! <end>
1,Jump.,<start> à¤à¤à¤²à¥. <end>
2,Jump.,<start> à¤à¥à¤¦à¥. <end>
3,Jump.,<start> à¤à¤²à¤¾à¤à¤. <end>
4,Hello!,<start> à¤¨à¤®à¤¸à¥à¤¤à¥à¥¤ <end>


Convert English Sentences to Lowercase

In [10]:
# Convert English sentences to lowercase
dataset["English"] = dataset["English"].str.lower()



dataset.head()

,English,Hindi
0,help!,<start> à¤¬à¤à¤¾à¤! <end>
1,jump.,<start> à¤à¤à¤²à¥. <end>
2,jump.,<start> à¤à¥à¤¦à¥. <end>
3,jump.,<start> à¤à¤²à¤¾à¤à¤. <end>
4,hello!,<start> à¤¨à¤®à¤¸à¥à¤¤à¥à¥¤ <end>


Import Required Libraries

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Create English Tokenizer



In [12]:
english_tokenizer = Tokenizer(filters='')
english_tokenizer.fit_on_texts(dataset["English"])

#Check vocabulary size:

english_vocab_size = len(english_tokenizer.word_index) + 1
print("English Vocabulary Size:", english_vocab_size)

English Vocabulary Size: 102950


Create Hindi Tokenizer

In [13]:
hindi_tokenizer = Tokenizer(filters='')
hindi_tokenizer.fit_on_texts(dataset["Hindi"])

#Check vocabulary size:

hindi_vocab_size = len(hindi_tokenizer.word_index) + 1
print("Hindi Vocabulary Size:", hindi_vocab_size)

Hindi Vocabulary Size: 95398


In [14]:
#Convert Sentences into Integer Sequences
#English
encoder_sequences = english_tokenizer.texts_to_sequences(dataset["English"])
#Hindi
decoder_sequences = hindi_tokenizer.texts_to_sequences(dataset["Hindi"])

Find Maximum Sequence Length

In [15]:
max_encoder_length = max(len(seq) for seq in encoder_sequences)
max_decoder_length = max(len(seq) for seq in decoder_sequences)

print("Maximum English Length:", max_encoder_length)
print("Maximum Hindi Length:", max_decoder_length)

Maximum English Length: 398
Maximum Hindi Length: 420


Pad the Sequences


In [16]:
#Encoder Input
encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_length,
    padding="post"
)

Decoder Input

In [17]:
decoder_input_data = pad_sequences(
    decoder_sequences,
    maxlen=max_decoder_length,
    padding="post"
)

In [18]:
#Display Shapes
print("Encoder Input Shape:", encoder_input_data.shape)
print("Decoder Input Shape:", decoder_input_data.shape)

Encoder Input Shape: (127375, 398)
Decoder Input Shape: (127375, 420)


Save the Trained Model

Save the trained Multivariate LSTM model for future predictions.

In [19]:
print("Encoder Input Shape:", encoder_input_data.shape)
print("Decoder Input Shape:", decoder_input_data.shape)

Encoder Input Shape: (127375, 398)
Decoder Input Shape: (127375, 420)


Create Decoder Target Data

In [20]:
import numpy as np

decoder_target_data = np.zeros(
    (len(decoder_input_data), max_decoder_length, 1),
    dtype="int32"
)

for i, seq in enumerate(decoder_sequences):
    for t in range(1, len(seq)):
        decoder_target_data[i, t-1, 0] = seq[t]

Check Shapes

In [21]:
print("Encoder Input Shape :", encoder_input_data.shape)
print("Decoder Input Shape :", decoder_input_data.shape)
print("Decoder Target Shape:", decoder_target_data.shape)

Encoder Input Shape : (127375, 398)
Decoder Input Shape : (127375, 420)
Decoder Target Shape: (127375, 420, 1)


Split into Training and Testing Data

In [22]:
from sklearn.model_selection import train_test_split

(
    encoder_train,
    encoder_test,
    decoder_train,
    decoder_test,
    target_train,
    target_test
) = train_test_split(
    encoder_input_data,
    decoder_input_data,
    decoder_target_data,
    test_size=0.2,
    random_state=42
)

Display Shapes

In [23]:
print("Encoder Train :", encoder_train.shape)
print("Encoder Test  :", encoder_test.shape)

print("Decoder Train :", decoder_train.shape)
print("Decoder Test  :", decoder_test.shape)

print("Target Train  :", target_train.shape)
print("Target Test   :", target_test.shape)

Encoder Train : (101900, 398)
Encoder Test  : (25475, 398)
Decoder Train : (101900, 420)
Decoder Test  : (25475, 420)
Target Train  : (101900, 420, 1)
Target Test   : (25475, 420, 1)


Set Model Parameters

In [24]:
embedding_dim = 256
latent_dim = 256

print("English Vocabulary Size:", english_vocab_size)
print("Hindi Vocabulary Size:", hindi_vocab_size)
print("Embedding Dimension:", embedding_dim)
print("LSTM Units:", latent_dim)

English Vocabulary Size: 102950
Hindi Vocabulary Size: 95398
Embedding Dimension: 256
LSTM Units: 256


mport Deep Learning Libraries

In [25]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

Build the Seq2Seq LSTM Model

In [26]:
#Encoder Input Layer
# Encoder
encoder_inputs = Input(shape=(max_encoder_length,), name="encoder_inputs")

Encoder Embedding Layer



In [27]:
encoder_embedding = Embedding(
    input_dim=english_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

Encoder LSTM

The encoder returns the final hidden state and cell state.

In [28]:
encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [29]:
#Decoder Input Layer
decoder_inputs = Input(shape=(max_decoder_length,), name="decoder_inputs")

Decoder Embedding Layer

In [30]:
decoder_embedding = Embedding(
    input_dim=hindi_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)(decoder_inputs)

Decoder LSTM

The decoder receives the encoder's final states.

In [31]:
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

Output Dense Layer

In [32]:
decoder_dense = Dense(
    hindi_vocab_size,
    activation="softmax",
    name="output_layer"
)

decoder_outputs = decoder_dense(decoder_outputs)

Create the Model

In [33]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

Compile the Model

In [34]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Display Model Summary

In [35]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 398)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 420)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 398, 256)  │ 26,355,200 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 398)       │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 420, 256)  │ 24,421,888 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    525,312 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 420,      │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 420,       │ 24,517,286 │ decoder_lstm[0][… │
│ (Dense)             │ 95398)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 76,344,998 (291.23 MB)

 Trainable params: 76,344,998 (291.23 MB)

 Non-trainable params: 0 (0.00 B)

Train the Model

In [36]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

In [37]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [38]:
print(model)

<Functional name=functional_1, built=True>


In [39]:
print("Encoder Input :", encoder_input_data.shape)
print("Decoder Input :", decoder_input_data.shape)
print("Decoder Target:", decoder_target_data.shape)

Encoder Input : (127375, 398)
Decoder Input : (127375, 420)
Decoder Target: (127375, 420, 1)


In [40]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 398)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 420)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 398, 256)  │ 26,355,200 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 398)       │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 420, 256)  │ 24,421,888 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    525,312 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 420,      │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 420,       │ 24,517,286 │ decoder_lstm[0][… │
│ (Dense)             │ 95398)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 76,344,998 (291.23 MB)

 Trainable params: 76,344,998 (291.23 MB)

 Non-trainable params: 0 (0.00 B)

Train the Seq2Seq Model

In [42]:
pip install transformers sentencepiece torch

In [43]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Model downloaded successfully!")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model downloaded successfully!


In [44]:
text = "Good morning. How are you?"

inputs = tokenizer(text, return_tensors="pt", padding=True)
translated = model.generate(**inputs)

print("English :", text)
print("Hindi   :", tokenizer.decode(translated[0], skip_special_tokens=True))

English : Good morning. How are you?
Hindi   : आप कैसे हैं?


In [45]:
sentences = [
    "I love India.",
    "What is your name?",
    "Where are you going?",
    "Good morning.",
    "Thank you very much.",
    "I am learning Artificial Intelligence.",
    "Today is a beautiful day."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)
    hindi = tokenizer.decode(translated[0], skip_special_tokens=True)

    print(f"English : {text}")
    print(f"Hindi   : {hindi}")
    print("-"*50)

English : I love India.
Hindi   : मैं भारत से प्यार करता हूँ.
--------------------------------------------------
English : What is your name?
Hindi   : आपका Windows Live कूटशब्द क्या है?
--------------------------------------------------
English : Where are you going?
Hindi   : तुम कहाँ जा रहे हो?
--------------------------------------------------
English : Good morning.
Hindi   : सुप्रभात.
--------------------------------------------------
English : Thank you very much.
Hindi   : बहुत बहुत धन्यवाद.
--------------------------------------------------
English : I am learning Artificial Intelligence.
Hindi   : मैं कलावादी कौशल सीख रहा हूँ.
--------------------------------------------------
English : Today is a beautiful day.
Hindi   : आज एक खूबसूरत दिन है ।
--------------------------------------------------


In [46]:
!pip -q install transformers sentencepiece sacremoses accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 18.8 MB/s eta 0:00:00


Download the model

In [47]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model downloaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model downloaded successfully!


Translate English → Hindi

In [56]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = "What is your name?"

# Tokenize with source language
inputs = tokenizer(text, return_tensors="pt")
inputs["forced_bos_token_id"] = tokenizer.convert_tokens_to_ids("hin_Deva")

# Generate translation
translated_tokens = model.generate(**inputs)

# Decode
translated_text = tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)[0]

print("English :", text)
print("Hindi   :", translated_text)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

English : What is your name?
Hindi   : तुम्हारा नाम क्या है?


Create a Reusable Translation Function

In [57]:
def translate_english_to_hindi(text):
    tokenizer.src_lang = "eng_Latn"

    encoded = tokenizer(text, return_tensors="pt")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("hin_Deva"),
        max_length=128
    )

    return tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

Test the Function

In [58]:
sentences = [
    "Hello, how are you?",
    "What is your name?",
    "Where are you going?",
    "I love India.",
    "Good morning.",
    "Thank you very much.",
    "Artificial Intelligence is changing the world.",
    "Today is a beautiful day.",
    "My name is Sugumar.",
    "I am learning Deep Learning."
]

for s in sentences:
    print("="*60)
    print("English :", s)
    print("Hindi   :", translate_english_to_hindi(s))

English : Hello, how are you?
Hindi   : हैलो, आप कैसे हैं?
English : What is your name?
Hindi   : तुम्हारा नाम क्या है?
English : Where are you going?
Hindi   : तुम कहाँ जा रहे हो?
English : I love India.
Hindi   : मुझे भारत बहुत पसंद है।
English : Good morning.
Hindi   : शुभ दोपहर.
English : Thank you very much.
Hindi   : बहुत बहुत धन्यवाद.
English : Artificial Intelligence is changing the world.
Hindi   : कृत्रिम बुद्धिमत्ता दुनिया बदल रही है।
English : Today is a beautiful day.
Hindi   : आज एक सुंदर दिन है।
English : My name is Sugumar.
Hindi   : मेरा नाम सुगुमार है।
English : I am learning Deep Learning.
Hindi   : मैं गहरी शिक्षा सीख रहा हूँ।


Interactive Translator

In [59]:
while True:
    text = input("\nEnter English sentence (type 'exit' to quit): ")

    if text.lower() == "exit":
        print("Translator closed.")
        break

    print("Hindi :", translate_english_to_hindi(text))


Enter English sentence (type 'exit' to quit): I am very happy
Hindi : मैं बहुत खुश हूँ

Enter English sentence (type 'exit' to quit): cash on delivery
Hindi : वितरण पर नकदी

Enter English sentence (type 'exit' to quit): exit
Translator closed.


Save the Model

In [60]:
model.save_pretrained("english_hindi_nllb")
tokenizer.save_pretrained("english_hindi_nllb")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
